# Netflix Media Consumption Analysis: Interactive Visuals

This notebook builds interactive charts and a geographic Folium map from the processed Netflix dataset.

## Goals
- Create interactive genre and timeline visualizations with Plotly.
- Build a country-level production map with Folium.
- Save the map artifact for sharing in `notebooks/images`.

## 1. Environment Setup

Import libraries and set project-relative paths for portability.

In [1]:
from pathlib import Path
import os
import sys

import folium
import pandas as pd
import plotly.express as px

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

from src.preprocess import clean_data, feature_engineering, load_data

## 2. Load and Prepare Data

Load the Netflix titles dataset and apply preprocessing pipeline functions.

In [2]:
data_path = Path("../data/raw/netflix_titles.csv")
df = load_data(data_path)
df = clean_data(df)
df = feature_engineering(df)

print(f"Dataset ready: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset ready: 8,807 rows x 17 columns


## 3. Interactive Bar Chart: Top 10 Genres

Display the most frequent primary genres using an interactive horizontal bar chart.

In [3]:
top_genres = df["primary_genre"].value_counts().head(10).reset_index()
top_genres.columns = ["Genre", "Count"]

fig = px.bar(
    top_genres.sort_values("Count"),
    x="Count",
    y="Genre",
    orientation="h",
    color="Count",
    color_continuous_scale="Viridis",
    title="Top 10 Genres on Netflix (Interactive)",
)
fig.update_layout(xaxis_title="Count", yaxis_title="Genre")
fig.show()

## 4. Interactive Timeline: Content Added per Year

Plot the annual trend of titles added to Netflix over time.

In [4]:
yearly = df["year_added"].value_counts().sort_index().reset_index()
yearly.columns = ["Year", "Count"]

fig2 = px.line(
    yearly,
    x="Year",
    y="Count",
    markers=True,
    title="Content Added to Netflix Over Time",
)
fig2.update_layout(xaxis_title="Year", yaxis_title="Number of Titles")
fig2.show()

## 5. Interactive Map: Countries Producing Netflix Content

Map title counts by country using capital city coordinates as representative points.

In [5]:
cities_path = Path("../data/worldcities.xlsx")
cities_df = pd.read_excel(cities_path)

cities_df.head()

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.6870,139.7495,Japan,JP,JPN,Tōkyō,primary,37785000.0,1392685764
1,Jakarta,Jakarta,-6.1750,106.8275,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1360771077
2,Delhi,Delhi,28.6100,77.2300,India,IN,IND,Delhi,admin,32226000.0,1356872604
3,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1156237133
4,Mumbai,Mumbai,19.0761,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1356226629


In [6]:
capital_coords = (
    cities_df[cities_df['capital'] == 'primary']
    .groupby('country')[['lat', 'lng']]
    .mean()
    .round(4)
    .reset_index()
    .set_index('country')
    .T
    .to_dict()
)


In [7]:
world_map = folium.Map(location=[20, 0], zoom_start=2, tiles="cartodb positron")

country_counts = (
    df["country"]
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .replace("Unknown", pd.NA)
    .dropna()
    .value_counts()
)

for country, count in country_counts.items():
    if country in capital_coords:
        coords = [capital_coords[country]["lat"], capital_coords[country]["lng"]]
        folium.CircleMarker(
            location=coords,
            radius=max(3, min(15, count / 50)),
            popup=f"{country}: {count}",
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.55,
        ).add_to(world_map)

world_map

In [8]:
output_map_path = Path("images/netflix_map.html")
output_map_path.parent.mkdir(parents=True, exist_ok=True)
world_map.save(output_map_path)

print(f"Interactive map saved to: {output_map_path.resolve()}")

Interactive map saved to: C:\Users\a12u\OneDrive\Desktop\Courses\Own Projects\data-analysis-mini-projects\media-consumption-analysis\notebooks\images\netflix_map.html
